# Pandas Essentials

## A full year of Mars weather

You have been promoted. Instead of single sols or a week of data, you now have a full dataset of Mars weather spanning thousands of sols. Over 5,000 rows of temperatures, wind speeds, pressures, and instrument readings.

You *could* process this with the raw Python tools from the previous notebooks -- csv.DictReader, list comprehensions, manual aggregation. But you would quickly drown in boilerplate. Calculating an average temperature per Martian month would take 30 lines of fiddly code.

**pandas** is a library that was built to solve exactly this problem. It gives us a data structure called a DataFrame -- essentially a spreadsheet in Python -- with powerful tools for filtering, grouping, aggregating, and visualising data. It is the most important library in the Python data ecosystem, and learning it well is the single biggest upgrade you can make to your data engineering toolkit.

In [ ]:
%pip install -q pandas matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## Loading data

One line. That is all it takes to load a CSV into a DataFrame.

In [ ]:
df = pd.read_csv("../data/mars_weather_2024.csv")
df.head()

Compare that with the 10+ lines of csv.DictReader code we wrote in the previous notebook. pandas read the file, parsed the headers, inferred column types, and gave us a structured table. All in one line.

Let's get oriented with the data.

In [ ]:
# Shape: (rows, columns)
print(f"Shape: {df.shape}")
print(f"That's {df.shape[0]:,} rows and {df.shape[1]} columns")

In [ ]:
# .info() gives us column names, types, and non-null counts
df.info()

In [ ]:
# .describe() gives us summary statistics for numeric columns
df.describe()

Already, `.describe()` has told us the mean, standard deviation, min, max, and quartiles for every numeric column. That would have been dozens of lines of manual code.

A few things to notice:
- The `count` row shows how many non-null values each column has. If count is less than the total rows, there are missing values.
- Check the `min` and `max` rows for outliers -- do any values look suspicious?

In [ ]:
# Look at the last few rows
df.tail()

In [ ]:
# Column names
print(df.columns.tolist())

In [ ]:
# Data types
print(df.dtypes)

## Selecting columns and rows

A DataFrame is made of **Series** -- each column is a Series. We can select columns by name and rows by position or condition.

In [ ]:
# Select a single column (returns a Series)
temps = df["min_temp_c"]
print(type(temps))
print(temps.head())

In [ ]:
# Select multiple columns (returns a DataFrame)
temp_data = df[["sol", "min_temp_c", "max_temp_c"]]
temp_data.head()

In [ ]:
# Select rows by position with .iloc
print("First row:")
print(df.iloc[0])
print("\nRows 10-14:")
df.iloc[10:15]

## Filtering: boolean indexing

This is where pandas starts to feel like a superpower. We can filter rows using conditions, and the syntax reads almost like English.

In [ ]:
# Find extremely cold sols (min temp below -90C)
extreme_cold = df[df["min_temp_c"] < -90]
print(f"Found {len(extreme_cold)} extremely cold sols")
extreme_cold.head(10)

What just happened? `df["min_temp_c"] < -90` creates a Series of `True`/`False` values, one per row. When we use this as an index into `df`, pandas returns only the rows where the condition is `True`. It is the same logic as the list comprehension filters from Notebook 2, but far more concise.

In [ ]:
# Multiple conditions: use & (and) and | (or), with parentheses
danger_days = df[
    (df["min_temp_c"] < -90) | 
    (df["wind_speed_ms"] > 40)
]
print(f"Danger days: {len(danger_days)}")
danger_days[["sol", "min_temp_c", "wind_speed_ms", "instrument_status"]].head(10)

In [ ]:
# .query() is an alternative syntax that some find more readable
warnings = df.query("instrument_status == 'warning'")
print(f"Sols with warnings: {len(warnings)}")
warnings.head()

In [ ]:
# Count values in a categorical column
df["instrument_status"].value_counts()

## Handling missing data

Real datasets always have gaps. Sensors fail, transmissions get corrupted, fields are left blank. pandas represents missing values as `NaN` (Not a Number), and gives us tools to detect, remove, or fill them.

In [ ]:
# How much data is missing?
print("Missing values per column:")
print(df.isna().sum())
print(f"\nTotal missing: {df.isna().sum().sum()}")
print(f"Percentage missing: {df.isna().sum().sum() / df.size * 100:.1f}%")

In [ ]:
# Look at rows with missing min_temp
missing_temp = df[df["min_temp_c"].isna()]
print(f"Rows missing min_temp: {len(missing_temp)}")
missing_temp.head()

In [ ]:
# Strategy 1: Drop rows with any missing values
df_clean = df.dropna()
print(f"Original: {len(df)} rows")
print(f"After dropna: {len(df_clean)} rows")
print(f"Dropped: {len(df) - len(df_clean)} rows")

In [ ]:
# Strategy 2: Drop only rows where specific columns are missing
df_temp_clean = df.dropna(subset=["min_temp_c", "max_temp_c"])
print(f"After dropping rows with missing temps: {len(df_temp_clean)} rows")

In [ ]:
# Strategy 3: Fill missing values
# Fill missing wind speed with the median
median_wind = df["wind_speed_ms"].median()
print(f"Median wind speed: {median_wind:.1f} m/s")

df_filled = df.copy()
df_filled["wind_speed_ms"] = df_filled["wind_speed_ms"].fillna(median_wind)

print(f"Missing wind readings before: {df['wind_speed_ms'].isna().sum()}")
print(f"Missing wind readings after:  {df_filled['wind_speed_ms'].isna().sum()}")

Which strategy to use depends on the situation. Dropping rows is safest but loses data. Filling with the median preserves rows but introduces synthetic values. There is no universal right answer -- it depends on what you are trying to do with the data.

## Handling duplicates

Our dataset intentionally contains some duplicate rows (a common real-world problem when data is retransmitted or pipelines run twice). Let's find and remove them.

In [ ]:
# Check for duplicates
dupes = df.duplicated()
print(f"Duplicate rows: {dupes.sum()}")

# Show some duplicates
df[dupes].head()

In [ ]:
# Remove duplicates
df_deduped = df.drop_duplicates()
print(f"Before: {len(df)} rows")
print(f"After:  {len(df_deduped)} rows")
print(f"Removed: {len(df) - len(df_deduped)} duplicate rows")

## The groupby revelation

This is the moment pandas justifies its existence. Suppose we want the average minimum temperature for each Martian "season" (let's define seasons by sol ranges). In raw Python, this requires:

1. Create a dict to hold temperature lists for each season
2. Loop through every row
3. Determine which season the sol belongs to
4. Append the temperature to the right list
5. Loop through the dict to calculate averages

Let's do it both ways.

In [ ]:
# The raw Python way (30+ lines)
import csv

season_temps_raw = {}

with open("../data/mars_weather_2024.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        sol_str = row["sol"]
        temp_str = row["min_temp_c"]
        
        if not sol_str or not temp_str:
            continue
        
        sol = int(sol_str)
        temp = float(temp_str)
        
        # Determine season (roughly based on Mars orbital period)
        season_num = ((sol - 1) % 668) // 167  # 4 seasons per Mars year
        season_names = ["Spring", "Summer", "Autumn", "Winter"]
        season = season_names[season_num]
        
        if season not in season_temps_raw:
            season_temps_raw[season] = []
        season_temps_raw[season].append(temp)

print("Raw Python approach:")
for season, temps in season_temps_raw.items():
    avg = sum(temps) / len(temps)
    print(f"  {season}: {avg:.1f}C (n={len(temps)})")

In [ ]:
# The pandas way (3 lines)
df_work = df_deduped.copy()
season_names = ["Spring", "Summer", "Autumn", "Winter"]
df_work["season"] = df_work["sol"].apply(lambda s: season_names[((s - 1) % 668) // 167])

df_work.groupby("season")["min_temp_c"].mean().round(1)

Same result. A fraction of the code. And the pandas version is not just shorter -- it is faster, handles missing values automatically, and produces a labelled output that is immediately useful.

This is why pandas exists. When people say "use the right tool for the job," they mean this kind of thing.

In [ ]:
# groupby with multiple aggregations
season_summary = df_work.groupby("season").agg(
    avg_min_temp=("min_temp_c", "mean"),
    avg_max_temp=("max_temp_c", "mean"),
    avg_wind=("wind_speed_ms", "mean"),
    max_wind=("wind_speed_ms", "max"),
    sol_count=("sol", "count"),
).round(1)

season_summary

In [ ]:
# Group by instrument status
status_summary = df_work.groupby("instrument_status").agg(
    count=("sol", "count"),
    avg_wind=("wind_speed_ms", "mean"),
    avg_min_temp=("min_temp_c", "mean"),
).round(1)

status_summary

## .apply(): custom transformations

Sometimes the built-in aggregations are not enough and we need custom logic. `.apply()` lets us run any function on each row or each value in a column.

In [ ]:
# Apply a function to a column
def classify_temperature(temp):
    """Classify a temperature reading."""
    if pd.isna(temp):
        return "unknown"
    if temp < -80:
        return "extreme_cold"
    elif temp < -60:
        return "cold"
    elif temp < -40:
        return "mild"
    else:
        return "warm"

df_work["temp_class"] = df_work["min_temp_c"].apply(classify_temperature)
df_work["temp_class"].value_counts()

In [ ]:
# Apply a function to each row (axis=1)
def compute_temp_range(row):
    """Calculate the temperature range for a sol."""
    if pd.isna(row["min_temp_c"]) or pd.isna(row["max_temp_c"]):
        return None
    return row["max_temp_c"] - row["min_temp_c"]

df_work["temp_range"] = df_work.apply(compute_temp_range, axis=1)
print(f"Average daily temperature range: {df_work['temp_range'].mean():.1f}C")
df_work[["sol", "min_temp_c", "max_temp_c", "temp_range"]].head()

## Creating new columns

We have already seen this above, but it is worth highlighting: creating derived columns is one of the most common pandas operations.

In [ ]:
# Vectorised operations are much faster than .apply()
df_work["min_temp_f"] = (df_work["min_temp_c"] * 9/5) + 32
df_work["max_temp_f"] = (df_work["max_temp_c"] * 9/5) + 32

# Pressure in millibars
df_work["pressure_mbar"] = df_work["pressure_pa"] / 100

df_work[["sol", "min_temp_c", "min_temp_f", "pressure_pa", "pressure_mbar"]].head()

Notice we did not use `.apply()` here. Arithmetic on whole columns is called a **vectorised operation**, and it is significantly faster than applying a function row by row. Use vectorised operations whenever you can.

## Sorting

Straightforward but essential.

In [ ]:
# Coldest sols
df_work.nsmallest(5, "min_temp_c")[["sol", "earth_date", "min_temp_c", "wind_speed_ms"]]

In [ ]:
# Windiest sols
df_work.nlargest(5, "wind_speed_ms")[["sol", "earth_date", "wind_speed_ms", "instrument_status"]]

## Basic plotting

pandas integrates with matplotlib to produce quick visualisations. These are not publication-quality charts, but they are invaluable for understanding your data.

In [ ]:
# Line plot: temperature over time
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_work["sol"], df_work["min_temp_c"], alpha=0.5, linewidth=0.5, label="Min temp")
ax.plot(df_work["sol"], df_work["max_temp_c"], alpha=0.5, linewidth=0.5, label="Max temp")
ax.set_xlabel("Sol")
ax.set_ylabel("Temperature (C)")
ax.set_title("Mars Surface Temperature Over Time")
ax.legend()
plt.tight_layout()
plt.show()

You should be able to see the seasonal variation in the temperature data -- the sinusoidal pattern of Martian seasons.

In [ ]:
# Histogram: distribution of minimum temperatures
fig, ax = plt.subplots(figsize=(10, 4))
df_work["min_temp_c"].hist(bins=40, ax=ax, edgecolor="black", alpha=0.7)
ax.set_xlabel("Minimum Temperature (C)")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Daily Minimum Temperatures")
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: wind speed vs temperature
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df_work["wind_speed_ms"], df_work["min_temp_c"], alpha=0.2, s=5)
ax.set_xlabel("Wind Speed (m/s)")
ax.set_ylabel("Min Temperature (C)")
ax.set_title("Wind Speed vs Minimum Temperature")
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: average temperature by season
fig, ax = plt.subplots(figsize=(8, 4))
season_avg = df_work.groupby("season")["min_temp_c"].mean()
season_avg.plot(kind="bar", ax=ax, color=["#2ecc71", "#e74c3c", "#f39c12", "#3498db"])
ax.set_xlabel("Season")
ax.set_ylabel("Average Min Temperature (C)")
ax.set_title("Average Minimum Temperature by Martian Season")
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Instrument status distribution
fig, ax = plt.subplots(figsize=(6, 4))
df_work["instrument_status"].value_counts().plot(kind="bar", ax=ax, edgecolor="black")
ax.set_xlabel("Status")
ax.set_ylabel("Count")
ax.set_title("Instrument Status Distribution")
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

---

## Exercises

Use the `df` DataFrame (the original, loaded from `mars_weather_2024.csv`) for these exercises.

### Exercise 1: Exploration

1. How many rows and columns does the dataset have?
2. What is the mean atmospheric pressure?
3. What is the maximum wind speed recorded?
4. How many unique instrument statuses are there?

In [ ]:
# Your code here


### Exercise 2: Filtering

1. Find all sols where the wind speed exceeded 35 m/s. How many are there?
2. Find all sols where the instrument was offline AND the minimum temperature was below -70C.
3. What percentage of all sols had a UV index above 8?

In [ ]:
# Your code here


### Exercise 3: Missing data

1. Which column has the most missing values?
2. Create a cleaned version of the DataFrame with all rows containing missing pressure or wind data removed.
3. How many rows remain?

In [ ]:
# Your code here


### Exercise 4: Groupby analysis

1. Group the data by `instrument_status` and calculate the mean and max wind speed for each status.
2. Create a "mars_month" column by dividing the sol number into groups of 100 (sol 1-100 = month 1, 101-200 = month 2, etc.). Then find the average min temperature per month.
3. Which Mars month had the coldest average temperature?

In [ ]:
# Your code here


### Exercise 5: Plotting

Create the following visualisations:

1. A histogram of wind speeds (use 30 bins)
2. A line plot showing atmospheric pressure over time (by sol)
3. A scatter plot of max temperature vs min temperature (do they correlate?)

In [ ]:
# Your code here


### Exercise 6: Putting it all together

Write a complete analysis that:

1. Loads the data and removes duplicates
2. Creates a "danger_level" column: "high" if wind > 25 or temp < -80, "medium" if wind > 15 or temp < -70, otherwise "low"
3. Uses groupby to find the count and average wind speed for each danger level
4. Plots a bar chart of the danger level counts
5. Prints a summary sentence: "X% of sols were classified as high danger."

In [ ]:
# Your code here


---

## Summary

This notebook introduced pandas, the library that turns Python into a serious data analysis tool:

- **`pd.read_csv()`** loads data in one line
- **`.head()`, `.info()`, `.describe()`** give you an instant overview of your data
- **Boolean indexing** and **`.query()`** filter rows with readable conditions
- **`.isna()`, `.fillna()`, `.dropna()`** handle missing values
- **`.drop_duplicates()`** removes duplicate rows
- **`.groupby()` + `.agg()`** replaces dozens of lines of manual aggregation
- **`.apply()`** runs custom functions on columns or rows
- **`.plot()`** and matplotlib produce quick visualisations

The big revelation: a single `groupby` line can replace 30 lines of raw Python. This is not about laziness. It is about expressing your intent clearly and letting the library handle the mechanics. Code that says `df.groupby('season')['min_temp_c'].mean()` is more readable, more correct, and more maintainable than a hand-rolled loop.

You now have the Python foundations to work with real data. From here, the path leads to databases, data pipelines, cloud infrastructure, and all the other tools in the data engineering arsenal. But everything builds on what you have learned in these four notebooks.